# Construyendo un Dashboard con Streamlit y Plotly
## Plantilla didáctica — Dataset "tips"

**Objetivo:** aprender a transformar gráficos de Plotly (que ya conoces de pandas/notebooks) en un **dashboard interactivo** con Streamlit.

**Plan:**
1. Explorar el dataset `tips` con pandas (repaso rápido).
2. Construir cada gráfico por separado, viéndolo aquí mismo en el notebook (`fig.show()`).
3. Entender la 'traducción' de cada pieza a Streamlit (widgets, `st.metric`, `st.plotly_chart`).
4. Ensamblar todo en un archivo `tips_app.py` (la app completa).
5. Probarla en tu computadora (`streamlit run tips_app.py`).
6. Subir el proyecto a **GitHub** y publicarlo en **Streamlit Community Cloud** con un enlace público permanente.

>  **Idea clave de Streamlit:** un script de Streamlit se ejecuta **de arriba a abajo, completo**, cada vez que el usuario mueve un widget (un slider, un selectbox...). No hay "callbacks" mágicos — es Python normal, leído en orden.

## 0. Instalar librerías

In [ ]:
!pip install streamlit plotly pandas statsmodels -q


## 1. El dataset `tips`

Plotly trae incorporado el clásico dataset de propinas de un restaurante (no hace falta subir ni descargar nada).

Columnas:
- `total_bill`: monto total de la cuenta (/$)
- `tip`: propina dejada (/$)
- `sex`, `smoker`, `day`, `time`: características de la mesa
- `size`: número de personas en la mesa

In [ ]:
import pandas as pd
import plotly.express as px

df = px.data.tips()
df.head()


,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


### Preparar columnas (lo mismo que harás en la app)

Traducimos categorías al español y creamos una columna calculada `% propina`. Esto es **pandas puro** — nada de Streamlit todavía.

In [ ]:
df["Fumador"] = df["smoker"].map({"Yes": "Sí", "No": "No"})
df["Sexo"]    = df["sex"].map({"Female": "Mujer", "Male": "Hombre"})
df["Día"]     = df["day"].map({"Thur": "Jue", "Fri": "Vie", "Sat": "Sáb", "Sun": "Dom"})
df["Momento"] = df["time"].map({"Lunch": "Almuerzo", "Dinner": "Cena"})
df["% propina"] = (df["tip"] / df["total_bill"] * 100).round(1)

ORDEN_DIAS = ["Jue", "Vie", "Sáb", "Dom"]
df.head()


,total_bill,tip,sex,smoker,day,time,size,Fumador,Sexo,Día,Momento,% propina
0,16.99,1.01,Female,No,Sun,Dinner,2,No,Mujer,Dom,Cena,5.9
1,10.34,1.66,Male,No,Sun,Dinner,3,No,Hombre,Dom,Cena,16.1
2,21.01,3.50,Male,No,Sun,Dinner,3,No,Hombre,Dom,Cena,16.7
3,23.68,3.31,Male,No,Sun,Dinner,2,No,Hombre,Dom,Cena,14.0
4,24.59,3.61,Female,No,Sun,Dinner,4,No,Mujer,Dom,Cena,14.7


## 2. ¿Por qué `@st.cache_data`?

En Streamlit, **todo el script se re-ejecuta de arriba a abajo** cada vez que el usuario toca un filtro. Si cargar/transformar los datos (lo que acabas de hacer arriba) tomara varios segundos, se repetiría en CADA interacción — muy lento.

La solución es envolver esa lógica en una función decorada con `@st.cache_data`:

```python
@st.cache_data
def cargar_datos():
    df = px.data.tips()
    # ... transformaciones ...
    return df

df = cargar_datos()
```

Streamlit ejecuta `cargar_datos()` **una sola vez** y reutiliza el resultado en las siguientes ejecuciones, mientras el código de la función no cambie.

## 3. Gráfico 1 — Torta de fumadores

Patrón típico: `value_counts()` para contar categorías, luego `px.pie`. Aquí lo vemos con `fig.show()`; en Streamlit cambiarás esa línea por `st.plotly_chart(fig, width="stretch")`.

In [ ]:
COLOR_SI_NO = {"Sí": "#1D9E75", "No": "#D85A30"}

conteo = df["Fumador"].value_counts().reset_index()
conteo.columns = ["Fumador", "Cuentas"]

fig = px.pie(conteo, names="Fumador", values="Cuentas", hole=0.55,
             color="Fumador", color_discrete_map=COLOR_SI_NO)
fig.update_traces(textinfo="percent+label")
fig.show()


## 4. Gráfico 2 — Barras agrupadas (día x momento x fumador)

`groupby().size()` para contar combinaciones, luego `px.bar` con `facet_col` para separar en paneles por "Momento".

In [ ]:
g = (df.groupby(["Día", "Momento", "Fumador"])
       .size().reset_index(name="Cuentas"))

fig = px.bar(g, x="Día", y="Cuentas", color="Fumador",
             facet_col="Momento", barmode="stack",
             color_discrete_map=COLOR_SI_NO,
             category_orders={"Día": ORDEN_DIAS})
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()


## 5. Gráfico 3 — Histograma del % de propina

`px.histogram` con `color` para comparar dos grupos superpuestos (`barmode="overlay"` + `opacity`).

In [ ]:
COLOR_SEXO = {"Mujer": "#7F77DD", "Hombre": "#378ADD"}

fig = px.histogram(df, x="% propina", color="Sexo", nbins=20,
                   opacity=0.75, barmode="overlay",
                   color_discrete_map=COLOR_SEXO)
fig.show()


## 6. Gráfico 4 — Boxplot de la cuenta total por día

`px.box` muestra mediana, cuartiles y outliers por categoría — ideal para comparar distribuciones.

In [ ]:
COLOR_DIA = {"Jue": "#378ADD", "Vie": "#7F77DD", "Sáb": "#BA7517", "Dom": "#D85A30"}

fig = px.box(df, x="Día", y="total_bill", color="Día",
             color_discrete_map=COLOR_DIA,
             category_orders={"Día": ORDEN_DIAS},
             labels={"total_bill": "Cuenta total ($)"})
fig.show()


## 7. Gráfico 5 — Dispersión con línea de tendencia

`px.scatter` con `trendline="ols"` ajusta una regresión lineal automáticamente (requiere `statsmodels`).

In [ ]:
fig = px.scatter(df, x="total_bill", y="tip", color="Sexo",
                 trendline="ols",
                 labels={"total_bill": "Cuenta total ($)", "tip": "Propina ($)"})
fig.show()


## 8. De Plotly/notebook a Streamlit: la "traducción"

Ya construiste los 5 gráficos. Para convertirlos en un dashboard interactivo, esta es la equivalencia que vas a aplicar:

| En el notebook | En Streamlit |
|---|---|
| `fig.show()` | `st.plotly_chart(fig, width="stretch")` |
| Variable fija, ej. `dia = "Vie"` | Valor devuelto por un widget, ej. `dia = st.multiselect(...)` |
| `print(x)` / mostrar una celda | `st.write(x)`, `st.metric(...)`, `st.dataframe(...)` |
| Filtrar el `df` manualmente cambiando código | Filtrar el `df` con los valores de los widgets (el código no cambia, solo los valores) |
| Una celda = un gráfico | Una sección del script = un gráfico, organizados con `st.columns()` |

**Widgets clave que usaremos:**
- `st.multiselect("Día", options=[...], default=[...])` → lista de valores elegidos
- `st.radio("Momento", options=[...])` → un valor elegido
- `st.slider("Tamaño", min_value=, max_value=, value=(a, b))` → tupla `(min, max)`
- `st.selectbox("Colorear por", [...])` → un valor elegido
- `st.metric("Etiqueta", valor, delta=...)` → tarjeta numérica
- `st.columns(n)` → divide la página en `n` columnas
- `st.expander("título")` → sección colapsable
- `st.stop()` → detiene la ejecución (útil si el filtro deja el DataFrame vacío)

## 9. Ensamblar la app completa

Ahora juntamos TODO lo anterior — carga cacheada, sidebar con filtros, KPIs, los 5 gráficos y la tabla — en un solo archivo `tips_app.py`.

Lee los comentarios del código: cada sección está numerada y explicada. Es la misma estructura que ya viste arriba, solo que ahora los valores vienen de widgets en vez de estar fijos.

In [ ]:
%%writefile tips_app.py
# tips_app.py
# ============================================================
#  DASHBOARD DIDÁCTICO — Dataset "tips" (propinas en un restaurante)
#  Ejecutar en local:  streamlit run tips_app.py
#  Publicar:           subir este repo a GitHub y desplegarlo en
#                      Streamlit Community Cloud (ver README.md)
#
#  Este script es una PLANTILLA para aprender Streamlit + Plotly.
#  Cada sección explica QUÉ hace y POR QUÉ, para que puedas
#  reutilizar el mismo patrón con tus propios datos.
#
#  Idea clave de Streamlit: el script se ejecuta de ARRIBA A ABAJO
#  cada vez que el usuario interactúa con un widget (un slider, un
#  selectbox, etc.). Por eso vas a ver que todo el "flujo" del
#  dashboard es simplemente código Python normal, leído de arriba
#  hacia abajo.
# ============================================================

import pandas as pd
import plotly.express as px
import streamlit as st

# ── 1. CONFIGURACIÓN DE LA PÁGINA ────────────────────────────
# st.set_page_config() debe ser la PRIMERA instrucción de Streamlit
# en el script. Define el título de la pestaña del navegador, el
# ícono y si el contenido usa todo el ancho ("wide") o queda
# centrado ("centered").
st.set_page_config(
    page_title="Tips Dashboard",
    page_icon="🍽️",
    layout="wide",
)

# ── 2. PALETA DE COLORES Y ESTILO COMPARTIDO ─────────────────
# Definir los colores como constantes evita repetir códigos hex
# y asegura que la misma categoría (ej. "Sí"/"No") tenga siempre
# el mismo color en TODOS los gráficos del dashboard.
COLOR_SI_NO = {"Sí": "#1D9E75", "No": "#D85A30"}
COLOR_SEXO  = {"Mujer": "#7F77DD", "Hombre": "#378ADD"}
COLOR_DIA   = {"Jue": "#378ADD", "Vie": "#7F77DD", "Sáb": "#BA7517", "Dom": "#D85A30"}
ORDEN_DIAS  = ["Jue", "Vie", "Sáb", "Dom"]

# Diccionario reutilizable con el estilo visual de Plotly: fondo,
# tipografía, márgenes y posición de la leyenda. Se "desempaqueta"
# con ** en cada fig.update_layout(**LAYOUT, ...).
LAYOUT = dict(
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Inter,Arial,sans-serif", size=12),
    margin=dict(t=40, b=40, l=50, r=20),
    legend=dict(orientation="h", y=-0.2),
)

# ── 3. CARGAR Y PREPARAR LOS DATOS ───────────────────────────
# @st.cache_data le dice a Streamlit: "ejecuta esta función UNA
# SOLA VEZ y guarda (cachea) el resultado en memoria". Como el
# script se vuelve a ejecutar completo cada vez que se mueve un
# filtro, sin esta línea recargaríamos y transformaríamos el
# dataset una y otra vez — innecesario y lento.
@st.cache_data
def cargar_datos():
    # px.data.tips() trae el dataset clásico de propinas ya
    # incluido en Plotly: no requiere descargar ni subir archivos.
    df = px.data.tips()

    # Traducimos/renombramos valores para que el dashboard se vea
    # en español. Esto es opcional, pero ayuda a la legibilidad
    # y es un buen ejemplo de "limpieza de datos" con .map().
    df["Fumador"] = df["smoker"].map({"Yes": "Sí", "No": "No"})
    df["Sexo"]    = df["sex"].map({"Female": "Mujer", "Male": "Hombre"})
    df["Día"]     = df["day"].map({"Thur": "Jue", "Fri": "Vie", "Sat": "Sáb", "Sun": "Dom"})
    df["Momento"] = df["time"].map({"Lunch": "Almuerzo", "Dinner": "Cena"})

    # Columna calculada: porcentaje de propina sobre la cuenta.
    # Esto es pandas normal — nada especial de Streamlit aquí.
    df["% propina"] = (df["tip"] / df["total_bill"] * 100).round(1)

    return df

df = cargar_datos()

# ── 4. SIDEBAR — FILTROS GLOBALES ────────────────────────────
# Todo lo que esté dentro de "with st.sidebar:" aparece en el
# panel lateral izquierdo. Cada widget (multiselect, radio, slider)
# DEVUELVE el valor elegido por el usuario, que guardamos en una
# variable normal de Python para usarla más abajo.
with st.sidebar:
    st.title("🍽️ Tips")
    st.caption("Filtros globales — afectan todos los gráficos")

    # multiselect → el usuario puede elegir VARIAS opciones a la vez
    dias = st.multiselect(
        "Día",
        options=ORDEN_DIAS,
        default=ORDEN_DIAS,
    )

    # radio → el usuario elige UNA sola opción
    momento = st.radio(
        "Momento",
        options=["Todos", "Almuerzo", "Cena"],
    )

    # slider de rango → devuelve una tupla (mínimo, máximo)
    tam_min, tam_max = st.slider(
        "Tamaño del grupo (personas)",
        min_value=int(df["size"].min()),
        max_value=int(df["size"].max()),
        value=(int(df["size"].min()), int(df["size"].max())),
    )

    st.divider()
    st.caption(f"Dataset completo: {len(df)} cuentas")

# ── 5. APLICAR LOS FILTROS (pandas puro) ─────────────────────
# Construimos una "máscara" booleana (una serie de True/False)
# combinando condiciones con & (Y lógico). Luego usamos esa
# máscara para quedarnos solo con las filas que la cumplen.
# Esta parte es 100% pandas — no tiene nada de Streamlit.
mask = (
    df["Día"].isin(dias) &
    df["size"].between(tam_min, tam_max)
)
if momento != "Todos":
    mask &= df["Momento"] == momento

dff = df[mask]  # dff = "dataframe filtrado"

# ── 6. ENCABEZADO ─────────────────────────────────────────────
st.title("Propinas en el restaurante")
st.caption(f"Mostrando {len(dff):,} de {len(df):,} cuentas con los filtros aplicados")

# Si el usuario filtra de forma que no queda ninguna fila, avisamos
# y detenemos la ejecución con st.stop(). Así evitamos errores al
# intentar graficar (o calcular promedios de) un DataFrame vacío.
if dff.empty:
    st.warning("No hay datos para los filtros seleccionados. Ajusta el panel lateral.")
    st.stop()

# ── 7. KPIs (TARJETAS DE MÉTRICAS) ───────────────────────────
# st.columns(n) crea n columnas de igual ancho una al lado de la
# otra. st.metric() muestra un número grande con una etiqueta
# arriba y, opcionalmente, una variación ("delta") debajo.
k1, k2, k3, k4 = st.columns(4)

cuentas       = len(dff)
propina_media = dff["tip"].mean()
pct_media     = dff["% propina"].mean()
pct_global    = df["% propina"].mean()  # referencia SIN filtros

k1.metric("Cuentas", f"{cuentas:,}")
k2.metric("Propina promedio", f"${propina_media:,.2f}")
k3.metric("Cuenta promedio", f"${dff['total_bill'].mean():,.2f}")
k4.metric("% propina promedio", f"{pct_media:.1f}%",
          delta=f"{pct_media - pct_global:+.1f} pts vs global")

st.divider()

# ── 8. FILA 1 — DOS GRÁFICOS LADO A LADO ─────────────────────
# st.columns([1, 2]) crea dos columnas donde la segunda es el
# doble de ancha que la primera. Cada "with col:" indica qué va
# dentro de esa columna.
col1, col2 = st.columns([1, 2])

with col1:
    st.subheader("¿Quiénes son fumadores?")
    # value_counts() cuenta cuántas veces aparece cada valor de
    # la columna — pandas puro, igual que en un notebook normal.
    conteo = dff["Fumador"].value_counts().reset_index()
    conteo.columns = ["Fumador", "Cuentas"]

    fig = px.pie(
        conteo, names="Fumador", values="Cuentas", hole=0.55,
        color="Fumador", color_discrete_map=COLOR_SI_NO,
    )
    fig.update_traces(textinfo="percent+label")
    fig.update_layout(**LAYOUT, showlegend=False)
    # st.plotly_chart() es el "puente" entre una figura de Plotly
    # y la página de Streamlit. width="stretch" hace que
    # el gráfico ocupe todo el ancho de su columna.
    st.plotly_chart(fig, width="stretch")

with col2:
    st.subheader("Cuentas por día y momento")
    # groupby + size() cuenta filas por combinación de columnas
    # (otra operación 100% pandas).
    g = (dff.groupby(["Día", "Momento", "Fumador"])
            .size().reset_index(name="Cuentas"))

    fig = px.bar(
        g, x="Día", y="Cuentas", color="Fumador",
        facet_col="Momento", barmode="stack",
        color_discrete_map=COLOR_SI_NO,
        category_orders={"Día": ORDEN_DIAS},
    )
    fig.update_layout(**LAYOUT)
    # Los "facets" de Plotly etiquetan cada panel como "Momento=Cena".
    # Esta línea se queda solo con la palabra después del "=".
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    st.plotly_chart(fig, width="stretch")

# ── 9. FILA 2 — DOS GRÁFICOS MÁS ─────────────────────────────
col3, col4 = st.columns(2)

with col3:
    st.subheader("Distribución del % de propina")
    fig = px.histogram(
        dff, x="% propina", color="Sexo", nbins=20,
        opacity=0.75, barmode="overlay",
        color_discrete_map=COLOR_SEXO,
    )
    fig.update_layout(**LAYOUT)
    st.plotly_chart(fig, width="stretch")

with col4:
    st.subheader("Cuenta total por día")
    fig = px.box(
        dff, x="Día", y="total_bill", color="Día",
        color_discrete_map=COLOR_DIA,
        category_orders={"Día": ORDEN_DIAS},
        labels={"total_bill": "Cuenta total ($)"},
    )
    fig.update_layout(**LAYOUT, showlegend=False)
    st.plotly_chart(fig, width="stretch")

# ── 10. FILA 3 — GRÁFICO INTERACTIVO CON SELECTBOX LOCAL ─────
# Este selectbox NO está en el sidebar: solo afecta a este gráfico.
# Es un buen ejemplo de la diferencia entre un "filtro global"
# (sidebar, afecta todo) y un "control local" (afecta solo una
# sección de la página).
st.subheader("Relación entre cuenta total y propina")

agrupar_por = st.selectbox(
    "Colorear por",
    ["Sexo", "Fumador", "Momento", "Día"],
    key="sel_color",  # 'key' único: obligatorio si hay varios selectbox
)

fig = px.scatter(
    dff, x="total_bill", y="tip", color=agrupar_por,
    trendline="ols",  # línea de tendencia (requiere el paquete statsmodels)
    labels={"total_bill": "Cuenta total ($)", "tip": "Propina ($)"},
)
fig.update_layout(**LAYOUT)
st.plotly_chart(fig, width="stretch")

# ── 11. TABLA DE DATOS (DESPLEGABLE) ─────────────────────────
# st.expander crea una sección colapsable: útil para no saturar
# la pantalla con la tabla completa todo el tiempo.
with st.expander("Ver datos filtrados"):
    columnas = ["Día", "Momento", "size", "total_bill", "tip", "% propina", "Sexo", "Fumador"]
    st.dataframe(
        dff[columnas].rename(columns={"size": "Personas"}),
        width="stretch",
        hide_index=True,
    )

# ════════════════════════════════════════════════════════════
# 🧪 EJERCICIOS PARA PRACTICAR
# (Bórralos cuando termines — son solo ideas para extender la app)
#
#  1. Agrega un widget st.checkbox("Solo cuentas grandes (size >= 4)")
#     y combínalo con la máscara de filtros (sección 5).
#
#  2. Agrega una quinta columna de KPI con la propina MÁXIMA
#     (dff["tip"].max()).
#
#  3. Cambia el gráfico de la sección 9 (col4) de px.box a
#     px.violin y compara cómo se ve la distribución.
#
#  4. Agrega un nuevo st.selectbox en la sección 10 para elegir
#     también la variable del eje X (por ejemplo, "size" en vez
#     de "total_bill").
# ════════════════════════════════════════════════════════════

Writing tips_app.py


## 10. Probar el dashboard en tu computadora

Antes de publicar, verifica que la app funciona en local. Desde la carpeta del proyecto:

```bash
pip install -r requirements.txt
streamlit run tips_app.py
```

Se abrirá `http://localhost:8501` en tu navegador. Cada vez que guardes `tips_app.py`, Streamlit ofrece recargar la página.

> Si estás trabajando en Colab, la celda de la sección 9 ya creó `tips_app.py`: descárgalo desde el panel de archivos (ícono de carpeta) para subirlo a GitHub. En el repositorio el archivo ya viene incluido.

## 11. Subir el proyecto a GitHub

Streamlit Community Cloud lee la app **directamente desde un repositorio de GitHub**. El repositorio necesita, como mínimo, estos archivos:

| Archivo | Para qué sirve |
|---|---|
| `tips_app.py` | La app (el script de la sección 9) |
| `requirements.txt` | Las librerías que la nube debe instalar (`streamlit`, `plotly`, `pandas`, `statsmodels`) |
| `README.md` | Descripción del proyecto e instrucciones |
| `.gitignore` | Evita subir archivos temporales (`__pycache__`, entornos virtuales, secretos) |
| `.streamlit/config.toml` | Tema visual de la app (opcional) |

**Opción A — Desde la web (sin instalar nada):**
1. En [github.com](https://github.com) crea un repositorio nuevo (botón **New**), por ejemplo `tips-dashboard`, visibilidad **Public**.
2. En el repositorio vacío elige **uploading an existing file**, arrastra los archivos y confirma con **Commit changes**.

**Opción B — Con git en la terminal:**
```bash
cd tips-dashboard
git init
git add .
git commit -m "Dashboard de propinas con Streamlit"
git branch -M main
git remote add origin https://github.com/TU_USUARIO/tips-dashboard.git
git push -u origin main
```

> **Ojo con `requirements.txt`:** en tu computadora instalaste las librerías a mano, pero el servidor de Streamlit parte vacío. Si falta una librería en ese archivo (por ejemplo `statsmodels`, que usa la línea de tendencia), la app publicada falla con `ModuleNotFoundError` aunque en local funcione.

## 12. Publicar en Streamlit Community Cloud

1. Entra a [share.streamlit.io](https://share.streamlit.io), haz clic en **Continue to sign-in** y luego en **Continue with GitHub**. Autoriza el acceso a tus repositorios.
2. En tu espacio de trabajo haz clic en **Create app** (arriba a la derecha) y, en **Deploy a public app from GitHub**, elige **Deploy now**.
3. Completa el formulario:
   - **Repository:** `TU_USUARIO/tips-dashboard`
   - **Branch:** `main`
   - **Main file path:** `tips_app.py`
   - **App URL (opcional):** un nombre a tu gusto, por ejemplo `tips-dashboard-tunombre`
4. Confirma con **Deploy**. La primera vez tarda unos minutos mientras se instalan las librerías de `requirements.txt`.

Al terminar tendrás un enlace público del tipo `https://tips-dashboard-tunombre.streamlit.app` que puedes abrir desde cualquier dispositivo y proyectar al presentar.

**Para actualizar la app** no hay que volver a desplegar: cada cambio que subas a GitHub (`git push`, o editar el archivo en la web) se refleja solo en la app publicada.

**Si algo falla:** abre **Manage app** (esquina inferior derecha de la app) para ver los registros. Los errores más comunes son una librería que falta en `requirements.txt` o un *Main file path* mal escrito.

## 🧪 Ejercicios para los estudiantes

Modifica `tips_app.py` (sección correspondiente entre comentarios `── N. ──`) para:

1. **Filtro nuevo:** agrega `st.checkbox("Solo cuentas grandes (size >= 4)")` en el sidebar y combínalo con la máscara de la sección 5.
2. **KPI nuevo:** agrega una quinta columna con la propina máxima (`dff["tip"].max()`).
3. **Otro tipo de gráfico:** cambia el boxplot de la sección 9 (`col4`) por `px.violin` y compara.
4. **Más interactividad:** agrega otro `st.selectbox` en la sección 10 para elegir la variable del eje X (por ejemplo `"size"` en vez de `"total_bill"`).
5. **Tu propio dataset:** reemplaza `px.data.tips()` por un CSV propio (`pd.read_csv(...)`) y adapta las columnas — el resto de la estructura (filtros, KPIs, gráficos) se mantiene igual.

Después de cada cambio, guarda el archivo y Streamlit se recargará solo (local). Cuando te guste el resultado, súbelo a GitHub y la app publicada se actualizará automáticamente.